# Multi-dataset Analysis: FIQA vs NFCorpus\n\nThis notebook compares retrieval performance and efficiency across multiple BEIR datasets,\nfocusing on FIQA and NFCorpus. It assumes that tuning has been run for each dataset so that\nthe following files exist under `../logs/`:\n\n- `tuning_fiqa.csv` and `tuning_fiqa_runs.csv`\n- `tuning_nfcorpus.csv` and `tuning_nfcorpus_runs.csv`\n\nUse `python scripts/run_experiments.py --mode tuning --datasets fiqa,nfcorpus` from the\nproject root to generate these logs.\n

In [ ]:
from __future__ import annotations\n\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom IPython.display import display\n\nproject_root = Path("..").resolve()\nlogs_dir = project_root / "logs"\n\ndef load_dataset_runs(dataset: str):\n    runs_path = logs_dir / f"tuning_{dataset}_runs.csv"\n    combined_path = logs_dir / f"tuning_{dataset}.csv"\n    if not runs_path.exists() or not combined_path.exists():\n        print(f"[WARN] Missing tuning logs for {dataset}.")\n        return None, None\n    runs_df = pd.read_csv(runs_path)\n    combined_df = pd.read_csv(combined_path)\n    # Derive mean latency and cost per run (averaged across queries and retrieval types)\n    agg = combined_df.groupby("run_id")["latency_s", "cost_tokens"].mean().reset_index()\n    runs_df = runs_df.merge(agg, on="run_id", how="left")\n    return runs_df, combined_df\n\nfiqa_runs, fiqa_combined = load_dataset_runs("fiqa")\nnf_runs, nf_combined = load_dataset_runs("nfcorpus")\n\ndisplay({"fiqa_runs_head": fiqa_runs.head() if fiqa_runs is not None else None})\ndisplay({"nfcorpus_runs_head": nf_runs.head() if nf_runs is not None else None})\n

In [ ]:
def pareto_frontier(df: pd.DataFrame) -> pd.DataFrame:\n    """Compute Pareto frontier where we maximize accuracy and minimize latency and cost."""\n    if df is None or df.empty:\n        return df\n    pts = df[["mean_accuracy", "latency_s", "cost_tokens"]].to_numpy()\n    is_pareto = np.ones(pts.shape[0], dtype=bool)\n    for i, p in enumerate(pts):\n        if not is_pareto[i]:\n            continue\n        # Any point that is strictly better in all dimensions dominates\n        dominates = (pts[:, 0] >= p[0]) & (pts[:, 1] <= p[1]) & (pts[:, 2] <= p[2]) & (\n            (pts[:, 0] > p[0]) | (pts[:, 1] < p[1]) | (pts[:, 2] < p[2])\n        )\n        is_pareto[dominates] = False\n    return df[is_pareto]\n\ndef plot_accuracy_latency_cost(df: pd.DataFrame, dataset: str):\n    if df is None or df.empty:\n        print(f"[INFO] No data to plot for {dataset}.")\n        return\n    frontier = pareto_frontier(df)\n\n    fig, ax = plt.subplots(figsize=(6, 4))\n    sc = ax.scatter(\n        df["mean_accuracy"],\n        df["latency_s"],\n        c=df["cost_tokens"],\n        cmap="viridis",\n        alpha=0.6,\n        label="all runs",\n    )\n    ax.scatter(\n        frontier["mean_accuracy"],\n        frontier["latency_s"],\n        c="red",\n        label="Pareto frontier",\n        marker="x",\n    )\n    ax.set_xlabel("mean_accuracy")\n    ax.set_ylabel("mean_latency_s")\n    ax.set_title(f"Accuracy vs Latency (color = cost) for {dataset}")\n    cbar = plt.colorbar(sc, ax=ax)\n    cbar.set_label("mean_cost_tokens")\n    ax.legend()\n    plt.show()\n\nplot_accuracy_latency_cost(fiqa_runs, "fiqa")\nplot_accuracy_latency_cost(nf_runs, "nfcorpus")\n

In [ ]:
# REM weight sensitivity analysis for FIQA (example)\n\nif fiqa_runs is None or fiqa_runs.empty:\n    print("[INFO] Skipping REM weight sensitivity: fiqa runs not available.")\nelse:\n    # Normalize metrics across runs\n    acc = fiqa_runs["mean_accuracy"].to_numpy()\n    lat = fiqa_runs["latency_s"].to_numpy()\n    cost = fiqa_runs["cost_tokens"].to_numpy()\n\n    def _minmax(x):\n        x = x.astype(float)\n        xmin, xmax = x.min(), x.max()\n        if xmax == xmin:\n            return np.ones_like(x)\n        return (x - xmin) / (xmax - xmin)\n\n    acc_n = _minmax(acc)\n    lat_n = _minmax(lat)\n    cost_n = _minmax(cost)\n\n    alphas = np.linspace(0.2, 0.8, 7)\n    betas = np.linspace(0.1, 0.5, 5)\n    # For simplicity, fix gamma = 1 - alpha - beta for each pair where that is positive\n\n    best_run_counts = {int(rid): 0 for rid in fiqa_runs["run_id"]}\n\n    for a in alphas:\n        for b in betas:\n            g = 1.0 - a - b\n            if g <= 0:\n                continue\n            rem_vals = a * acc_n + b * (1.0 - lat_n) + g * (1.0 - cost_n)\n            best_idx = int(rem_vals.argmax())\n            best_rid = int(fiqa_runs.iloc[best_idx]["run_id"])\n            best_run_counts[best_rid] += 1\n\n    best_summary = pd.DataFrame(\n        {\"run_id\": list(best_run_counts.keys()), \"num_best_weight_settings\": list(best_run_counts.values())}\n    ).sort_values("num_best_weight_settings", ascending=False)\n    display(best_summary.head(10))\n